# Stage 2 — Proposed Hybrid Model

**PMLDL 2026 project — Minimal Requirement 2.** A *single* model: non-learning
domain features feeding one gradient-boosting pipeline.

## Attribution

This notebook is our own composition, but it borrows two ideas from public
Kaggle material for the *Hull Tactical Market Prediction* competition:

| Borrowed | Source |
| --- | --- |
| Domain signal family — short-horizon mean reversion, inverse-volatility blending, volatility targeting | 4th place write-up, *"technical model, no learning, short-term mean reversion"* (competition write-ups page) |
| Pipeline architecture — lag/rolling temporal context, single LightGBM, Optuna tuned on mean Spearman across time-series folds | 61st place notebook, <https://www.kaggle.com/code/rafanikitas/hull-eda-training-pipeline> |
| Cross terms `U1`, `U2` | Hull starter notebook; also used in the 100th place write-up |

The 4th place author does not disclose the exact alpha. What is reproduced is
the *class* of indicator and the risk-management framing, rebuilt from scratch
in `src/features.py`. No author names, usernames or team identifiers appear
anywhere in this project.

Shared infrastructure (`src/`) is documented in `INTERFACE.md`. Seed is fixed at
42 project-wide; folds are the same `get_folds()` defaults used by all three
Stage 1 baselines, so every number below is directly comparable.

## Why combine these two specifically

The two sources pull in opposite directions, which is what makes the hybrid
interesting rather than arbitrary.

The 4th place solution demonstrates that this signal family works with **no
learning at all** — a rule-based mean-reversion alpha, auxiliary signals
combined by inverse-volatility weighting, and a volatility-targeting overlay.
Crucially, that author reports the overlay contributed more to the leaderboard
than any change to the alpha, and that most engineered features survived
*because they stabilised the allocation*, not because they predicted returns.

The 61st place solution demonstrates the opposite discipline: a **single**
LightGBM on temporal context beat the author's own CatBoost/XGBoost ensembles,
tuned on rank correlation rather than RMSE because exact magnitudes are not
learnable in this noise regime.

The gap between them is the opportunity. The 4th place combination weights are
fixed *a priori* — inverse-vol, no fitting, deliberately so, to avoid
overfitting. But the right weighting of a mean-reversion signal is plausibly
**state-dependent**: reversion pays in volatile or range-bound markets and
bleeds in persistent trends, which that author names as the strategy's inherent
limitation. Learning a state-dependent combination is exactly what a GBDT does
well. So:

> **Hypothesis.** Feed the 4th place *indicators* (computed with no learning)
> into the 61st place *pipeline*, and let the tree learn the regime-dependent
> weighting that inverse-vol weighting fixes by hand.

### One design decision worth flagging

The 61st pipeline applies lags and rolling windows to *anonymised* columns,
which is necessarily blind — there is no way to reason about whether a rolling
std of `V13` is meaningful. Our domain block is *already* a rolling transform of
the return series. Applying lag/rolling expansion on top of it would multiply
near-duplicate collinear columns and hand the tree a large set of features that
differ only by window overlap.

So the two blocks enter differently: **lag/rolling expansion on raw anonymised
columns only** (61st, unchanged), **domain features unexpanded** (4th place).
This is a deliberate deviation from a literal merge of the two recipes.

### Leakage

`forward_returns` at row *t* spans *t → t+1*. Every price-derived feature is
built from `past_returns()` = `forward_returns.shift(1)`, so no feature reads
its own row's future. Validation uses purged, embargoed folds. The last 180
`date_id`s are held out and never fitted on.

In [ ]:
# ============================ SETUP ============================
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from src import *          # shared interface, see INTERFACE.md

optuna.logging.set_verbosity(optuna.logging.WARNING)

set_seed()                 # SEED = 42, fixed project-wide
print("seed:", SEED, "| target:", TARGET)

# Trial budget. QUICK=True is for a fast smoke run; the reported results use
# QUICK=False. Either way the search is seeded, so runs are reproducible.
QUICK = False
N_TRIALS = 15 if QUICK else 120
TIMEOUT = 300 if QUICK else 3600

## 1. Data

`load_dataset()` sorts by `date_id`, trims the mostly-null early rows, and holds
out the last 180 `date_id`s as the public block. Identical call in every
notebook.

In [ ]:
hull = load_dataset()
print(hull)
print("train date_id:", hull.train[DATE_COL].min(), "->", hull.train[DATE_COL].max())
print("public date_id:", hull.public[DATE_COL].min(), "->", hull.public[DATE_COL].max())

## 2. Feature engineering

Three blocks, built on the **full** chronological frame so rolling windows stay
continuous across the train/public boundary — safe because every window looks
strictly backwards.

**Block A — domain features (4th place, no learning).** From
`src.features.add_price_features`, all derived from the lagged return series:

* *momentum* — cumulative return over 5/10/20/60 days;
* *realised volatility* — annualised std over 5/20/60 days;
* *volatility spreads* — `vol_5 - vol_60`, plus 5/60 and 20/60 ratios. The term
  structure of risk is what tells a calm market from a stressed one;
* *mean-reversion z-scores* — recent mean return over 3/5/10 days divided by
  20-day volatility, sign-flipped, so a positive value means "recently
  oversold". This is the 4th place alpha family;
* *drawdown* — distance from the 20-day equity-curve high.

**Block B — cross terms.** `U1 = I2 - I1` (term-structure spread) and
`U2 = M11 / mean(I2, I9, I7)` (rate-normalised market dynamic).

**Block C — temporal context (61st).** Lags 1/3/5/7/14/20 and rolling
mean/std over 2/5/10/20/60, applied to the same 14 anonymised columns the 61st
author selected — kept verbatim so the comparison against Baseline 02 isolates
the effect of Block A rather than a change of shortlist.

In [ ]:
# The 61st place shortlist, reproduced verbatim for comparability.
LAG_ROLL_COLUMNS = ["M4", "V13", "S5", "S2", "D2", "E19", "P7", "P6",
                    "P3", "P13", "P4", "P5", "M2", "V5"]
LAG_ROLL_COLUMNS = [c for c in LAG_ROLL_COLUMNS if c in hull.full.columns]

feat_df, FEATURES = build_features(
    hull.full,
    lag_roll_columns=LAG_ROLL_COLUMNS,   # Block C: raw anonymised columns only
    price_features=True,                 # Block A: 4th place domain signals
    cross_terms=True,                    # Block B
)

DOMAIN = [c for c in FEATURES if c.startswith(("mom_", "vol_", "revert_",
                                               "drawdown", "ret_1d"))]
print(f"total features: {len(FEATURES)}  (domain: {len(DOMAIN)}, "
      f"lag/roll: {sum('_lag_' in c or '_roll_' in c for c in FEATURES)})")
print("domain block:", DOMAIN)

# Sanity: no look-ahead column can reach the model.
assert not set(FEATURES) & set(LOOKAHEAD_COLS), "look-ahead column in feature list"
# Sanity: ret_1d really is the previous row's forward return.
_a = feat_df["ret_1d"].to_numpy()[1:]; _b = feat_df["forward_returns"].to_numpy()[:-1]
assert np.allclose(_a, _b, equal_nan=True), "ret_1d is not a clean 1-step lag"
print("leakage assertions passed")

In [ ]:
# Re-split on date_id, then impute with TRAIN-ONLY statistics.
cut = feat_df[DATE_COL].max() - PUBLIC_TEST_SIZE
train_df = feat_df[feat_df[DATE_COL] <= cut].reset_index(drop=True)
public_df = feat_df[feat_df[DATE_COL] > cut].reset_index(drop=True)

train_df, public_df = impute(train_df, public_df, columns=FEATURES)
assert train_df[FEATURES].isna().sum().sum() == 0
assert public_df[FEATURES].isna().sum().sum() == 0
print("train:", train_df.shape, "| public:", public_df.shape)

### Inverse-volatility blend as a reference signal

Before any learning, the 4th place recipe applied directly: combine the
mean-reversion signals by inverse-volatility weighting. This is the
**no-learning benchmark** the hybrid has to beat — if the GBDT cannot improve
on a weighted average of its own inputs, the learning step is not earning its
place.

In [ ]:
REVERT_COLS = [c for c in train_df.columns if c.startswith("revert_")]

blend_tr = blend_signals(train_df[REVERT_COLS]).fillna(0.0).to_numpy()
blend_pub = blend_signals(public_df[REVERT_COLS]).fillna(0.0).to_numpy()

no_learning = evaluate(
    train_df[TARGET].to_numpy(), blend_tr,
    weights=naive_allocation(blend_tr, k=0.5),
    forward_returns=train_df["forward_returns"].to_numpy(),
    risk_free_rate=train_df["risk_free_rate"].to_numpy(),
)
print("no-learning inverse-vol blend (in-sample, whole train period):")
print({k: round(v, 4) for k, v in no_learning.items() if isinstance(v, float)})

## 3. Validation

The same purged, embargoed folds as every other notebook: 4 expanding-window
splits, `purge=1` (the target horizon) and `embargo=20` trading days, so no
training row's target overlaps a validation block and no validation block sits
flush against correlated training rows.

In [ ]:
folds = get_folds(train_df)          # defaults — do not override
assert_no_leakage(folds)
display(describe_folds(train_df, folds))

## 4. Tuning

LightGBM with RMSE for early stopping but **Optuna maximising mean Spearman
rank correlation across folds** — the 61st place choice. The reasoning holds
here: predicting the exact magnitude of a de-meaned, winsorised excess return is
hopeless, whereas ordering days from weak to strong is not, and the ranking is
all the allocation rule consumes.

Search space follows the 61st notebook. Seeded sampler, so the search is
reproducible.

In [ ]:
X = train_df[FEATURES]
y = train_df[TARGET]


def cv_spearman(params, n_estimators_cap=None):
    """Mean Spearman IC across the shared purged folds."""
    scores, best_iters = [], []
    for fold in folds:
        X_tr, y_tr = X.iloc[fold.train_idx], y.iloc[fold.train_idx]
        X_va, y_va = X.iloc[fold.val_idx], y.iloc[fold.val_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric="rmse",
            callbacks=[lgb.early_stopping(200, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        scores.append(spearman_ic(y_va.to_numpy(), model.predict(X_va)))
        best_iters.append(model.best_iteration_ or params["n_estimators"])
    return float(np.mean(scores)), int(np.mean(best_iters))


def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": trial.suggest_int("n_estimators", 500, 7000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.07, log=True),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "num_leaves": trial.suggest_int("num_leaves", 32, 512),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "random_state": SEED,
        "verbosity": -1,
    }
    score, _ = cv_spearman(params)
    return score


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),   # reproducible search
    study_name="proposed_hybrid",
)
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT,
               show_progress_bar=False)

BEST_PARAMS = {
    "objective": "regression", "metric": "rmse", "subsample_freq": 1,
    "random_state": SEED, "verbosity": -1, **study.best_params,
}
print(f"trials: {len(study.trials)} | best mean Spearman IC: {study.best_value:.4f}")
print(BEST_PARAMS)

## 5. Cross-validated evaluation

One model per fold, refit with the tuned parameters. Allocation follows the
61st place discrete policy (`binary_allocation`): risk-free or fully invested,
nothing between. That author used it as a form of regularisation — it refuses to
treat small differences between two uncertain predictions as meaningful, and it
caps the damage from any single large forecast error. Stage 3 revisits sizing.

In [ ]:
fold_metrics, importances, oof = [], [], []

for fold in folds:
    X_tr, y_tr = X.iloc[fold.train_idx], y.iloc[fold.train_idx]
    X_va, y_va = X.iloc[fold.val_idx], y.iloc[fold.val_idx]
    val = train_df.iloc[fold.val_idx]

    model = lgb.LGBMRegressor(**BEST_PARAMS)
    model.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)], eval_metric="rmse",
              callbacks=[lgb.early_stopping(200, verbose=False),
                         lgb.log_evaluation(-1)])

    pred = model.predict(X_va)
    weights = binary_allocation(pred)          # 61st place discrete policy

    m = evaluate(y_va.to_numpy(), pred, weights=weights,
                 forward_returns=val["forward_returns"].to_numpy(),
                 risk_free_rate=val["risk_free_rate"].to_numpy())
    m["fold"] = fold.index
    fold_metrics.append(m)
    importances.append(model.feature_importances_)
    oof.append(pd.DataFrame({DATE_COL: val[DATE_COL].to_numpy(),
                             "y_true": y_va.to_numpy(),
                             "y_pred": pred, "weight": weights}))

cv_table = pd.DataFrame(fold_metrics)[
    ["fold", "spearman_ic", "rmse", "hit_rate", "modified_sharpe",
     "sharpe", "vol_ratio", "benchmark_sharpe", "mean_weight"]
]
display(cv_table.round(4))

cv_metrics = aggregate_folds([{k: v for k, v in m.items() if k != "fold"}
                              for m in fold_metrics])
print("\nmean Spearman IC: {spearman_ic_mean:.4f} (sd {spearman_ic_std:.4f})"
      .format(**cv_metrics))
print("mean modified Sharpe: {modified_sharpe_mean:.4f} (sd {modified_sharpe_std:.4f})"
      .format(**cv_metrics))
print("benchmark: {benchmark_sharpe_mean:.4f}".format(**cv_metrics))

Fold standard deviation is reported alongside every mean deliberately. With
four folds spanning different market regimes, a mean on its own hides whether a
model is consistently mediocre or wildly regime-dependent — and the latter is
precisely the failure mode the 4th place author warns about for mean-reversion
strategies.

## 6. Did the domain block earn its place?

The hypothesis was that the 4th place indicators add something the anonymised
lag/rolling block does not. Feature importance answers whether the tree uses
them; an ablation answers whether they help.

In [ ]:
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": np.mean(importances, axis=0),
})
imp["block"] = np.where(imp.feature.isin(DOMAIN), "domain (4th)",
                 np.where(imp.feature.str.contains("_lag_|_roll_"), "lag/roll (61st)",
                          "raw / cross"))
display(imp.sort_values("gain", ascending=False).head(20).reset_index(drop=True))

share = imp.groupby("block").agg(
    total_gain=("gain", "sum"), n_features=("feature", "size")).reset_index()
share["gain_share"] = (share.total_gain / share.total_gain.sum()).round(3)
share["gain_per_feature"] = (share.total_gain / share.n_features).round(1)
display(share)

In [ ]:
# Ablation on identical folds: domain block removed, everything else unchanged.
ABLATION = [c for c in FEATURES if c not in DOMAIN]
abl_scores = []
for fold in folds:
    m = lgb.LGBMRegressor(**BEST_PARAMS)
    m.fit(train_df.iloc[fold.train_idx][ABLATION], y.iloc[fold.train_idx],
          eval_set=[(train_df.iloc[fold.val_idx][ABLATION], y.iloc[fold.val_idx])],
          eval_metric="rmse",
          callbacks=[lgb.early_stopping(200, verbose=False),
                     lgb.log_evaluation(-1)])
    abl_scores.append(spearman_ic(y.iloc[fold.val_idx].to_numpy(),
                                  m.predict(train_df.iloc[fold.val_idx][ABLATION])))

print(f"with domain block:    {cv_metrics['spearman_ic_mean']:+.4f}")
print(f"without domain block: {np.mean(abl_scores):+.4f}")
print(f"delta:                {cv_metrics['spearman_ic_mean'] - np.mean(abl_scores):+.4f}")

## 7. Held-out public block

Final model refit on the whole training period with the tuned parameters, then
scored once on the 180 `date_id`s that were never touched. One shot, no
selection against this block.

In [ ]:
final_model = lgb.LGBMRegressor(**BEST_PARAMS)
final_model.fit(X, y)

pub_pred = final_model.predict(public_df[FEATURES])
pub_weights = binary_allocation(pub_pred)

public_metrics = evaluate(
    public_df[TARGET].to_numpy(), pub_pred, weights=pub_weights,
    forward_returns=public_df["forward_returns"].to_numpy(),
    risk_free_rate=public_df["risk_free_rate"].to_numpy(),
)
print({k: round(v, 4) for k, v in public_metrics.items() if isinstance(v, float)})

In [ ]:
# Allocation-rule comparison on the held-out block. Reported for context only
# — the headline result above uses the 61st place discrete policy, and nothing
# here is selected on.
rows = []
for name, w in [
    ("binary (61st)", binary_allocation(pub_pred)),
    ("naive k=50", naive_allocation(pub_pred, k=50)),
    ("vol-target", vol_target_allocation(pub_pred, public_df["vol_20"].to_numpy(),
                                         target_vol=0.12, k=50)),
    ("passive w=1", np.ones(len(pub_pred))),
]:
    r = modified_sharpe(w, public_df["forward_returns"].to_numpy(),
                        public_df["risk_free_rate"].to_numpy(),
                        return_components=True)
    r["rule"] = name
    r["mean_weight"] = float(np.mean(w))
    rows.append(r)
display(pd.DataFrame(rows)[["rule", "modified_sharpe", "sharpe", "vol_ratio",
                            "vol_penalty", "return_penalty", "mean_weight"]].round(4))

## 8. Save results

In [ ]:
MODEL_NAME = "hybrid_domain_lgbm"

save_result(model=MODEL_NAME, stage="proposed", metrics=cv_metrics, split="cv",
            params=BEST_PARAMS,
            notes="4th-place domain features + 61st-place single LightGBM; "
                  "Optuna on mean Spearman; binary allocation")
save_result(model=MODEL_NAME, stage="proposed", metrics=public_metrics,
            split="public", params=BEST_PARAMS,
            notes="refit on full train period, scored once on held-out 180 rows")

save_predictions(MODEL_NAME,
                 pd.concat(oof)[DATE_COL], pd.concat(oof)["y_true"],
                 pd.concat(oof)["y_pred"], pd.concat(oof)["weight"])

print("--- cross-validated ---")
display(compare(split="cv"))
print("--- held-out public block ---")
display(compare(split="public"))

## Conclusions

**What the hybrid is.** One LightGBM. No ensembling, no stacking, no online
retraining — those belong to Stage 3. The only structural change against
Baseline 02 is the domain feature block, which is why the ablation in section 6
is the honest measure of whether the idea worked.

**On reading these numbers.** `modified_sharpe` is our re-implementation of the
competition metric (volatility cliff at 120% of market volatility, quadratic
penalty for underperforming buy-and-hold), not the organisers' code. It is a
consistent internal yardstick for ranking our own models on identical folds; it
is not a leaderboard prediction. Compare models against each other and against
`benchmark_sharpe`, not against published competition scores.

**The standing caution from both sources.** The 61st place author found that
strong offline validation still failed to reproduce live uncertainty, and the
4th place author found that portfolio construction mattered more than the alpha.
Both point the same way: a small cross-validated edge in rank correlation is
fragile, and how it is turned into a position matters at least as much as the
model that produced it. That is the thread Stage 3 picks up.